In [ ]:
# Módulos

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
base_dir = Path().resolve().parents[1] # Nos situamos en la carpeta del proyecto
data_dir = base_dir / 'data/raw'

ruta = Path(data_dir / f'yt_stats.parquet')

if not ruta.exists():
    raise FileNotFoundError(f'No se encuentra la ruta: {ruta}')

df1 = pd.DataFrame(pd.read_parquet(ruta))

df1.head()

In [ ]:
import pandas as pd
import numpy as np

def procesar_impacto_youtube(df_original):
    def calcular_fila(row):
        score_total = 0
        encontrado_alguna_metrica = False
        
        for i in range(4):
            # Extraemos los valores
            v = row.get(f'video_{i}_video_statistics.viewCount', 0)
            l = row.get(f'video_{i}_video_statistics.likeCount', 0)
            c = row.get(f'video_{i}_video_statistics.commentCount', 0)
            
            # Convertimos a numérico de forma segura y rellenamos nulos con 0
            vals = pd.to_numeric([v, l, c], errors='coerce')
            v, l, c = np.nan_to_num(vals) 
            
            if v > 0 or l > 0 or c > 0:
                encontrado_alguna_metrica = True
                # Aplicamos la fórmula logarítmica
                sv = (0.5 * np.log10(v + 1)) + \
                     (0.3 * np.log10(l + 1)) + \
                     (0.2 * np.log10(c + 1))
                score_total += sv
        
        return score_total if encontrado_alguna_metrica else 0

    df_nuevo = pd.DataFrame()
    df_nuevo['id'] = df_original['id']
    
    # Aplicamos la lógica a cada fila
    df_nuevo['yt_score'] = df_original.apply(calcular_fila, axis=1)
    
    # Normalización final [0, 1]
    max_impacto = df_nuevo['yt_score'].max()
    if max_impacto > 0:
        df_nuevo['yt_score'] = df_nuevo['yt_score'] / max_impacto
        
    return df_nuevo

df = procesar_impacto_youtube(df1)
df.head()

In [ ]:
import plotly.express as px

fig = px.histogram(df, x="yt_score", nbins=50, 
                   title="Distribución de Impacto en YouTube (yt_score)",
                   labels={'yt_score': 'Score (0-1)', 'count': 'Cantidad de Juegos'},
                  )

fig.show()